# **===========================================================================================================================================================** #
# **Pantry To Table Ingredient Image Recognition Model Notebook** #
# **===========================================================================================================================================================** #
## This notebook trains, tunes and optimizes a YOLOv8m model for ingredient image recognition.

### What is YOLOv8m?
**Model Tier**: The "Medium" version of the YOLOv8 (You Only Look Once) architecture. </br>
</br>
**Optimal Balance**: Positioned between the faster, less accurate "Small" (v8s) and the highly accurate but slower "Large" (v8l) models.</br>
</br>
**Architecture**: A one-stage object detector that processes images in a single pass, predicting bounding boxes and class probabilities simultaneously.</br>
</br>
**Parameters**: Contains roughly 25.9 million parameters, making it complex enough for detailed recognition without being too heavy for consumer hardware.</br>

### Why are we using it for ingredient recognition?** 
**Hardware Compatibility**: It is perfectly sized for NVIDIA Blackwell/Lovelace GPUs. </br>
It utilizes the available VRAM effectively while providing near-instant inference speeds.</br>
</br>
**Class Complexity**: With 100+ different ingredients, a smaller model (v8n or v8s) would struggle.</br>
The Medium tier has the depth needed to distinguish these subtle differences.</br>
</br>
**High mAP (Mean Average Precision)**: It offers a significantly higher accuracy rate than the smaller models.</br>
This important because the output feeds into a recipe API—one wrong ingredient can ruin a recipe suggestion.</br>
</br>
**Training Efficiency**: Based on GPU/VRam used during development, v8m trains fast enough to allows quick </br>
turnover for multiple iterations and hyperparameter tuning.</br>
</br>
**Feature Richness**: It handles various scales well. It can detect a small items like a garlic clove and a large bag of </br>
flour in the same pantry photo with high confidence.</br>

#### This note books contains the required support methods for the yolov8m model:
- Develop
- Tune
- Train
- Test
- Analyze


In [ ]:
# Standard library imports

import os
import time
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

## **YOLOv8 Model**

### 1. Model Initialization

#### Start Processing Timer

In [ ]:
# ───────────────────────────────
# Constants
# ───────────────────────────────

MAX_IMAGE_SAMPLES      = 200      # Raw download pool per ingredient
SAMPLES_PER_INGREDIENT = 100      # Final "Clean" count for training

# For Tracking Notebook runtime/performance
notebook_start_time = time.time()

In [ ]:
# Using 'm' for 24GB VRAM - great balance for ingredient classification
model = YOLO('yolov8m.pt')

model.info()

### 2. Initial Training
#### The Final Training Phase of the model, using the optimized settings from the tuning phase.

**Target Duration**: 100 epochs - 100 iterations</br>
**Image Resolution**: The imgsz=640 scales all input images to 640×640 pixels.</br>
**Batch Size**: It processes 32 images at a time, which balances training speed with the memory capacity.</br>
**Hardware Acceleration**: Device=0, the workload is offloaded to the NVIDIA Blackwell GPU</br>
**Configuration Injection**: The cfg parameter loads the custom best_hyperparameters.yaml, so final run uses best hyperparameters.</br>
**Visual Analytics**: Setting plots=True automatically generate performance graphs, including Loss, Precision-Recall, and F1-Score curves, for model analysis.</br>

In [ ]:
print('─' * 32)
print("✅ Starting Initial Training...")
print('─' * 32)

results = model.train(
    data='pantrydata/data.yaml',
    epochs=100,
    patience=20,
    imgsz=640,
    batch=-1,
    device=0, # Blackwell GPU
    plots=True # Automatically generates F1, PR, and Loss plots
)

### 3. Hyperparameter Tuning (Automated AutoML)
**yolov8m** contains a tune() method which performs Hyperparameter tuning</br>
It replaces manual gradient tuning loops or manual hyperparameter tuning loops.</br>
(Hyperparemeter Tuning - automated process to find the best settings for your model before the final training begins.)</br>
</br>
**Resutls are saved** in the 'best_hyperparameters.yaml' in runs/detect/tune/</br>

- **Automated Optimization**: It runs 10 different iterations of the model to discover the best specific settings.
- **Data Source**: Uses the file path defined in data.yaml to locate the ingredient images and labels.
- **Tuning Duration**: Each of the 10 iterations runs for 30 "epochs"
- **Optimizer Selection**: Uses AdamW, a specific mathematical algorithm designed to update model weights efficiently during training.
- **Validation**: Sets val=True to test each iteration against validation image set to ensure the found settings actually work on new data.
- **Resource Management**: Setting plots=False and save=False, avoids saving unnecessary files for every single attempt.</br>
Only saving the final "best" configuration.
- **Final Output**: Once finished, it saves a best_hyperparameters.yaml file, which then use to train final model</br>
[Target Hardware - GPU VRam 24GB - Blackwell].

In [ ]:
# print('─' * 32)
# print("🚀 Starting Tuning yolov8m Model...")
# print('─' * 32)

# tune_results = model.tune(
#     data='pantrydata/data.yaml',
#     epochs=30, 
#     iterations=10, 
#     optimizer='AdamW', 
#     plots=False, 
#     save=False, 
#     val=True
# )

### 4. Final Training Run

In [ ]:
print('─' * 32)
print("✅ Starting Final Training...")
print('─' * 32)

results = model.train(
    data='pantrydata/data.yaml',
    epochs=100,
    patience=20,
    imgsz=640,
    batch=-1,
    device=0, # Blackwell GPU
    #cfg='runs/detect/tune/best_hyperparameters.yaml',
    plots=True # Automatically generates F1, PR, and Loss plots
)

### 5. Generate F1/AUC/Confusion Matrix Snapshots

In [ ]:
print('─' * 32)
print("📊 Generating Validation Snapshots...")
print('─' * 32)

metrics = model.val() # Uses the 'val' split from data.yaml

### 6. Model Analysis
#### Manual Metric Extraction And Plotting
Pull metric data from the automatically generated results.csv. </br>
Create visual plots for analysis.</br>
Model also automatically generates performance plots.</br>

In [ ]:
results_path = os.path.join(model.trainer.save_dir, 'results.csv')
df = pd.read_csv(results_path)
df.columns = [c.strip() for c in df.columns]

# Create custom plots for your project report
plt.figure(figsize=(12, 5))

# Plot mAP50 (Accuracy)
plt.subplot(1, 2, 1)
plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50', color='blue')
plt.title('Accuracy (AUC equivalent)')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(df['epoch'], df['train/cls_loss'], label='Train Class Loss', color='orange')
plt.plot(df['epoch'], df['val/cls_loss'], label='Val Class Loss', color='red')
plt.title('Classification Loss')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.savefig('custom_metrics_report.png')
plt.show()


print('─' * 32)
print()
print(f"Final Model Fitness: {results.fitness:.4f}")
print('─' * 32)
print()

In [ ]:
# Print Timing statistics
elapsed = time.time() - notebook_start_time
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)
print('─' * 32)
print('─' * 32)
print()
print(f'Total Notebook model training and tuning runtime: {hours}h {minutes}m {seconds}s')
print()
print('─' * 32)
print('─' * 32)
